# Day 11 — Advanced retrieval for harder questions: read around the hit, measured

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-11/LEARN.md`](../docs/day-11/LEARN.md).*

Day 10 ended with a held-out miss that no better word could fix: "My phone was stolen on a business trip. Who should I tell?" The weighted fusion found the right document, the Security Incident Reporting policy, but not the right section: the question describes an **event**, and the answer is the **procedure**, in the section next to one the retriever sent.

**Section expansion** uses the documents' structure: after retrieving, also send the sections around each hit, or its whole parent document. It is one of the most common "advanced" retrieval patterns (also called context expansion, small-to-big or parent-document retrieval), offered by RAG frameworks and managed search services. This notebook builds it from scratch, with:

1. an **off-switch** that gives back exactly Day 10's context,
2. a **per-query trace**: every section considered, why, and what happened to it,
3. a **permission check** on every added section, because a lookup by position isn't a search,
4. a **fair comparison**: an expansion sends more text, so it is measured not only against the top 3 but against a **control** that sends the same number of sections from further down the ranking,
5. **68 new held-out questions** (`questions-v3`).

**Every rule was committed before anything scored a held-out question.** [`evals/preregistration/day-11.md`](../evals/preregistration/day-11.md), the first commit on today's branch together with the new questions, holds the settings, the rule that chooses which sections to add, its dev values, what would be reported and what the decision needs. The notebook recomputes the rule on dev and stops if it gets another value.

Everything runs locally: no API key, no new packages and no new model (Day 04's embedding model), about a minute.

## 0. Setup

No new packages. The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) gains **Day 10's query analysis and vocabulary map** (`queries.py`: `analyse()`, `expand()` and the map), because today reproduces Day 10's baseline, which includes the two map retrievers. `tests/test_shared_code.py` checks it against the Day 10 notebook.

`EXPANSION` is the **off-switch**: what a pipeline would read from its configuration on every request. Everything measured below passes its setting explicitly, so flipping the switch changes only the trace in section 3 and the traces saved in section 10.

In [1]:
import dataclasses
import hashlib
import json
import math
import platform
import statistics
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
    import tokenizers
except ImportError as error:
    raise ImportError("Day 11 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.answering import (ABSTAIN, BLOCKED, DIRECTORY, MAX_CONTEXT_WORDS, SENTENCE, TOP_K,  # Day 06's
                                        ExtractiveReader, Pipeline, RagAnswer, assemble, check_citations,   # pipeline
                                        count_words)
from rag_from_scratch.bm25 import BM25Index                              # Day 01's BM25
from rag_from_scratch.chunking import section_chunks                     # Day 03's chunker
from rag_from_scratch.embeddings import Embedder                         # Day 04's model
from rag_from_scratch.evaluation import (TIE, compare, first_relevant, hit, load_queries, measure,  # Day 07's harness
                                         outcome, sign_test, tie_margin, top_score, validate, wilson)
from rag_from_scratch.fusion import ALPHA, weighted                      # Day 08's fusion
from rag_from_scratch.index import Result, VectorIndex, sync             # Day 05's index
from rag_from_scratch.ingest import ingest                               # Day 02's pipeline
from rag_from_scratch.queries import VOCABULARY, defined_abbreviations, expand   # Day 10's map (new in src/)

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__} | "
      f"tokenizers {tokenizers.__version__}")

Python 3.13.18 | numpy 2.4.6 | onnxruntime 1.31.0 | tokenizers 0.23.3


In [2]:
K_VALUES = (1, 3, 5, 10)
EVALS = REPO_ROOT / "evals"
V2 = {"questions": EVALS / "questions-v2.jsonl", "manifest": EVALS / "manifest-v2.json"}    # Day 10's 122 queries
V3 = {"questions": EVALS / "questions-v3.jsonl", "manifest": EVALS / "manifest-v3.json"}    # + today's 68 new ones
DAY10_BASELINE = EVALS / "baselines" / "day-10-retrieval-v2.json"   # Day 10's accepted run: today's reference
BASELINE_FILE = EVALS / "baselines" / "day-11-retrieval-v3.json"     # today's accepted run, committed and reviewed
PREREGISTRATION = EVALS / "preregistration" / "day-11.md"            # today's rules, committed before any held-out run
OUT_DIR = REPO_ROOT / "outputs" / "day-11"                           # this run's results; outputs/ is git-ignored

HITS = TOP_K                  # 3: every context starts with the weighted fusion's top 3 (Day 06's TOP_K, not tuned)
BUDGET = MAX_CONTEXT_WORDS    # 300 words, whole chunks (Day 06's budget, not tuned)
SETTINGS = ("previous", "next", "neighbours", "parent")   # what an expansion may add around each hit
EXPANSION = "neighbours"      # THE OFF-SWITCH: "off" sends the three hits only, exactly Day 10's top 3
DENSE_GATE = 0.62             # Day 07's measured gate on the dense top score (evals/baselines/day-07-retrieval.json)
READER_THRESHOLD = 0.63       # Day 07's measured reader threshold (same file)

# What evals/preregistration/day-11.md commits to. Section 3.1 recomputes the rule on dev and stops if it gives
# anything else.
PREREGISTERED = {"setting": "neighbours",
                 "dev": {"off": (88, 115.6), "previous": (89, 182.2), "next": (89, 179.1), "neighbours": (90, 223.1),
                         "parent": (89, 245.6)}}
prereg = PREREGISTRATION.read_text(encoding="utf-8")
assert "**The rule chooses neighbours.**" in prereg and "| neighbours | 90 | 223.1 |" in prereg \
    and "| off | 88 | 115.6 |" in prereg, "PREREGISTERED must match evals/preregistration/day-11.md"
assert EXPANSION in ("off",) + SETTINGS

## 1. The same queries, the same snapshot, Day 10's five retrievers

A comparison is only fair if everything except the new step stays the same. So, before changing anything, this section loads Day 10's 122 labelled queries (`questions-v2`), checks the documents against their snapshot, rebuilds Day 10's five retrievers (BM25, dense retrieval, the weighted fusion, and BM25 and the weighted fusion with Day 10's map) and checks that they **reproduce Day 10's accepted baseline**, query by query. Day 10's other baseline, on v1, also holds RRF and the two reranked hybrids; nothing builds on them, so they aren't rebuilt today (the Day 10 notebook's own test still checks that file on every push).

In [3]:
manifest2 = json.loads(V2["manifest"].read_text(encoding="utf-8"))
queries2 = load_queries(V2["questions"])
documents, _ = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
by_cid = {c.chunk_id: c for c in chunks}
problems2 = validate(queries2, chunks, manifest2)
v2_sha256 = hashlib.sha256(V2["questions"].read_bytes()).hexdigest()
if v2_sha256 != manifest2["questions_sha256"]:
    problems2.insert(0, f"{V2['questions'].name} changed but its manifest didn't")
print(f"{len(queries2)} labelled queries (version {manifest2['version']}) | {len(chunks)} chunks | label problems: "
      f"{len(problems2)}" + "".join(f"\n  {p}" for p in problems2))
assert not problems2, "The labels don't match the documents: run the Day 07 notebook and review them first."

embedder = Embedder(REPO_ROOT)     # Day 04's model: downloaded once (133 MB), SHA-256 checked on every run
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
sync(index, chunks, embedder.embed)   # the snapshot's chunks, embedded now (Day 05's sync into an empty index)
rag = Pipeline(index, embedder)       # Day 06's retrieve(): as the user, permission filter before the ranking
bm25 = {user: BM25Index({c.chunk_id: c.text for c in chunks if set(c.allowed_groups) & groups})
        for user, groups in DIRECTORY.items()}
ENTRIES = VOCABULARY + defined_abbreviations(chunks)    # Day 10's reviewed map (measured, not adopted)


def bm25_search(user: str, text: str) -> list[tuple[str, float]]:
    # Day 07's BM25: the user's own index, full ranking. Only chunks that share a word with the text come back.
    return [(h.id, h.score) for h in bm25[user].search(text, len(chunks))]


def dense_search(user: str, text: str) -> list[tuple[str, float]]:
    # Day 07's dense retrieval: every chunk the user may read, ranked by cosine similarity
    return [(r.id, r.score) for r in rag.retrieve(user, text, len(chunks))]


NAMES = ["BM25", "dense", "weighted", "BM25+map", "weighted+map"]   # Day 10's five retrievers on v2
rankings = {name: {} for name in NAMES}      # retriever -> query id -> FULL ranking, (chunk_id, score) best first


def add_rankings(qs) -> None:
    for q in qs:
        lexical, dense = bm25_search(q.user, q.question), dense_search(q.user, q.question)
        mapped = bm25_search(q.user, expand(q.question, ENTRIES).text)   # Day 10: the map on BM25's side only
        rankings["BM25"][q.id], rankings["dense"][q.id], rankings["BM25+map"][q.id] = lexical, dense, mapped
        rankings["weighted"][q.id] = weighted(lexical, dense, ALPHA)     # the default retriever since Day 08
        rankings["weighted+map"][q.id] = weighted(mapped, dense, ALPHA)


add_rankings(queries2)
print(f"Day 10's five retrievers on {len(queries2)} queries")

122 labelled queries (version 2) | 44 chunks | label problems: 0
tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified
onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


Day 10's five retrievers on 122 queries


In [4]:
day10 = json.loads(DAY10_BASELINE.read_text(encoding="utf-8"))
DATASETS = {2: {"questions": V2["questions"].relative_to(REPO_ROOT).as_posix(), "version": manifest2["version"],
                "sha256": v2_sha256, "snapshot": manifest2["snapshot"]["fingerprint"]}}
MAP_SETTINGS = {"code": "expand() in the Day 10 notebook", "expands": "BM25's query only (rule on dev)",
                "map": [[list(phrases), list(terms)] for phrases, terms, _ in ENTRIES],
                "matching": "whole phrases, case-insensitive; the documents' terms are appended, nothing removed",
                "routing": "an exact value (no ?, at most 4 words, a digit, @, # or ://) is not expanded"}
SETTINGS_OF = {   # Day 10's five retrievers exactly as Day 10 described them, so compare() checks the settings too
    "BM25": {"code": "rag_from_scratch.bm25 (Day 01)", "k1": bm25["alice"].k1, "b": bm25["alice"].b,
             "indexed_text": "chunk text with its Title > Section header",
             "permissions": "one index of each user's readable chunks (filter before the ranking)"},
    "dense": {"code": "rag_from_scratch.answering.Pipeline.retrieve (Day 06)", "model": embedder.model["repo"],
              "revision": embedder.revision, "query_instruction": True,
              "indexed_text": "chunk text with its Title > Section header",
              "permissions": "metadata filter before the ranking"},
    "weighted": {"code": "weighted() in the Day 08 notebook", "fuses": ["BM25", "dense"], "alpha_dense": ALPHA,
                 "normalisation": "min-max per retriever and query; 0 from a retriever that didn't return the chunk",
                 "alpha_rule": "most dev queries in the top 3, then the higher dev MRR@10, then closest to 0.5",
                 "inputs": "full rankings, each already permission-filtered", "ties": "chunk id"},
    "BM25+map": {**MAP_SETTINGS, "searches": "BM25 (Day 01, the user's own index) on the expanded question"},
    "weighted+map": {**MAP_SETTINGS, "fuses": ["BM25+map", "dense"], "alpha_dense": ALPHA,
                     "dense_query": "the original question"}}


def entry(name: str, q) -> dict:
    # One query, one retriever, in the baseline's format: the rank of the first labelled chunk, how close that is to a
    # tie, the top 3 and the best score
    ranked = rankings[name][q.id]
    return {"first_relevant": first_relevant(ranked, q.answer_chunks) if q.answerable else None,
            "margin": tie_margin(ranked, q.answer_chunks) if q.answerable else None,
            "top3": [cid for cid, _ in ranked[:HITS]], "top_score": top_score(ranked)}


def metrics_of(names, selections) -> dict:
    return {f"{kind}, {name}": {k: round(v, 4) if isinstance(v, float) else v
                                for k, v in measure(rankings[name], selection).items()}
            for kind, selection in selections.items() for name in names}


SELECTIONS2 = {"questions": [q for q in queries2 if q.kind == "question" and q.answerable],
               "lookups": [q for q in queries2 if q.kind == "lookup"]}
# Day 10's accepted file, with today's settings, ranks and metrics for its five retrievers in place of its own
reproduced = {**day10, "dataset": DATASETS[2], "retrievers": {name: SETTINGS_OF[name] for name in NAMES},
              "queries": {q.id: {name: entry(name, q) for name in NAMES} for q in queries2},
              "metrics": metrics_of(NAMES, SELECTIONS2)}
changes10, ties10 = compare(reproduced, day10)
same_top3 = all(reproduced["queries"][q.id][n]["top3"] == day10["queries"][q.id][n]["top3"] for q in queries2 for n in NAMES)
print(f"{', '.join(NAMES)} against {DAY10_BASELINE.relative_to(REPO_ROOT).as_posix()}: "
      + (f"{len(changes10)} changes" + "".join(f"\n  {c}" for c in changes10) if changes10 else
         f"no changes ({len(queries2)} queries x 5 retrievers: every rank, best score and metric; top 3 lists "
         f"{'identical' if same_top3 else 'DIFFERENT'})"))
for t in ties10:
    print(f"  near tie, not a change: {t}")

BM25, dense, weighted, BM25+map, weighted+map against evals/baselines/day-10-retrieval-v2.json: no changes (122 queries x 5 retrievers: every rank, best score and metric; top 3 lists identical)


Same labels, same snapshot, same five rankings: every difference below comes from today's change.

### 1.1 New held-out questions: `questions-v3`

Day 10 used v2's 41 test questions to decide, so they count as used now (`evals/README.md`, rules 4 and 8). [`evals/questions-v3.jsonl`](../evals/questions-v3.jsonl) keeps v2's 122 queries exactly (the 41 become dev; each note says so) and adds **68 new test questions**, more than v2's 41, because a held-out comparison on 29 answerable questions has wide intervals. They were written after the rules were frozen and committed with them, before any system ran on them. About half describe the situation first ("I'm still in my probation period. Can I work remotely?"), the way people ask. The pre-registration says what the author knew while writing them.

In [5]:
manifest3 = json.loads(V3["manifest"].read_text(encoding="utf-8"))
queries3 = load_queries(V3["questions"])
problems3 = validate(queries3, chunks, manifest3)
v3_sha256 = hashlib.sha256(V3["questions"].read_bytes()).hexdigest()
if v3_sha256 != manifest3["questions_sha256"]:
    problems3.insert(0, f"{V3['questions'].name} changed but its manifest didn't")
by_qid = {q.id: q for q in queries3}
used = [q.id for q in queries2 if q.split == "test"]
# v3 keeps v2's 122 queries exactly (text, user, labels); only the used test questions changed split (and say so)
same122 = all(q3 == dataclasses.replace(q2, split=q3.split, note=q3.note) for q2, q3 in zip(queries2, queries3)) and \
    [q.id for q in queries3[:122] if q.split != by_qid[q.id].split or q.id in used and q.split != "dev"] == [] and \
    all(by_qid[qid].note.endswith("so dev from v3 on)") for qid in used)
NEW = [q for q in queries3 if q.split == "test"]
print(f"{len(queries3)} labelled queries (version {manifest3['version']}): v2's {len(queries2)} "
      f"({'identical, its ' + str(len(used)) + ' test questions now dev' if same122 else 'CHANGED'}) + "
      f"{len(NEW)} new test questions ({sum(q.answerable for q in NEW)} answerable, "
      f"{sum(not q.answerable for q in NEW)} not), same snapshot | label problems: {len(problems3)}"
      + "".join(f"\n  {p}" for p in problems3))
assert not problems3 and same122 and manifest3["snapshot"] == manifest2["snapshot"]
assert {q.origin for q in NEW} == {"day11-new"} and all(q.kind == "question" for q in NEW)

190 labelled queries (version 3): v2's 122 (identical, its 41 test questions now dev) + 68 new test questions (52 answerable, 16 not), same snapshot | label problems: 0


## 2. The stolen phone: the answer is next door

Where is the answer, and what did the retriever send?

In [6]:
SECTIONS = defaultdict(list)     # document -> its sections, in their order in the document (Day 03's chunks)
for c in chunks:
    SECTIONS[c.doc_id].append(c.chunk_id)
POSITION = {cid: (doc, i) for doc, ids in SECTIONS.items() for i, cid in enumerate(ids)}
PHONE, LOCKOUT = by_qid["d10-a24"], by_qid["d10-a12"]


def where(cid: str) -> str:
    doc, i = POSITION[cid]
    return f"section {i + 1} of {len(SECTIONS[doc])}"


for q in (PHONE, LOCKOUT):
    print(f"{q.id} {q.question}\n  the answer: {q.answer_chunks[0]} ({where(q.answer_chunks[0])})\n"
          f"  the weighted fusion's top 6:")
    for rank_, (cid, score) in enumerate(rankings["weighted"][q.id][:6], start=1):
        print(f"    {rank_}  {score:.3f}  {cid:<56} {where(cid):<16}"
              + ("<- the answer" if cid in q.answer_chunks else "sent" if rank_ <= HITS else ""))
    doc = POSITION[q.answer_chunks[0]][0]
    print(f"  {doc}, in order: " + " | ".join(cid.split("#")[1] for cid in SECTIONS[doc]) + "\n")

d10-a24 My phone was stolen on a business trip. Who should I tell?
  the answer: security-incident-reporting#how-to-report (section 2 of 4)
  the weighted fusion's top 6:
    1  1.000  travel-policy#safety-while-travelling                    section 5 of 5  sent
    2  0.654  security-incident-reporting#after-you-report             section 4 of 4  sent
    3  0.568  security-incident-reporting#what-not-to-do               section 3 of 4  sent
    4  0.499  security-incident-reporting#what-counts-as-an-incident   section 1 of 4  
    5  0.469  travel-policy#claiming-travel-costs                      section 4 of 5  
    6  0.452  security-incident-reporting#how-to-report                section 2 of 4  <- the answer
  security-incident-reporting, in order: what-counts-as-an-incident | how-to-report | what-not-to-do | after-you-report

d10-a12 What happens if I type my password wrong too many times?
  the answer: password-and-mfa-standard#account-lockout (section 4 of 4)
  the weighted fu

The weighted fusion found the right document: three of its top four are sections of the Security Incident Reporting policy. But it ranked the sections that describe incidents (*After you report*, *What not to do*, *What counts as an incident*) above the one that says what to do (*How to report*, 6th), because the question describes an event, not a procedure. *How to report* is **section 2 of 4, right before hit 3**. The lockout question has the same shape for a different reason, a vocabulary gap: its answer, *Account lockout*, is the section right after hit 2.

Day 03 split the documents at their headings, so every chunk knows its document and its position in it. The ranking doesn't use that structure.

## 3. Section expansion: read around the hit

`expand_sections()` keeps the weighted fusion's three hits and then, hit by hit, adds the sections a **setting** names around each one:

| Setting | Adds around each hit |
|---|---|
| `off` | nothing: exactly the top 3 (the off-switch) |
| `previous` | the section just before it |
| `next` | the section just after it |
| `neighbours` | the section before it, then the one after it |
| `parent` | the hit's other sections, in document order: its whole parent document |

Three rules keep it honest:

1. **Hits first.** The top 3 is always sent (it always fits: no chunk has more than 75 words), so an expansion can add to Day 10's context but never lose any of it.
2. **Day 06's budget.** At most 300 words, whole sections; a section that doesn't fit is skipped and the next one tried.
3. **Every added section is permission-checked.** Finding a neighbour is a lookup by position, not a search: the search's permission filter never sees it (section 6).

Each call returns a **trace**: every section it considered, why, and what happened to it.

In [7]:
@dataclass(frozen=True)
class Step:
    chunk: str                 # the section considered
    why: str                   # "hit 2", "previous of hit 3", "parent of hit 1", ...
    score: float | None        # its fused score, if the retriever ranked it
    words: int
    decision: str              # "sent", "already sent", "not readable" or "over budget"


@dataclass(frozen=True)
class Context:
    question: str
    user: str
    setting: str
    sent: tuple[str, ...]      # what the reader gets, in this order: the hits, then what was added around them
    words: int
    trace: tuple[Step, ...]    # every section considered and what happened to it: the per-query trace


def around(cid: str, setting: str) -> list[tuple[str, str]]:
    # What a setting adds around one hit: a lookup in the document's structure, not a search
    doc, i = POSITION[cid]
    ids = SECTIONS[doc]
    before, after = ids[max(0, i - 1):i], ids[i + 1:i + 2]
    return {"off": [],
            "previous": [(x, "previous") for x in before],
            "next": [(x, "next") for x in after],
            "neighbours": [(x, "previous") for x in before] + [(x, "next") for x in after],
            "parent": [(x, "parent") for x in ids if x != cid]}[setting]


def may_read(cid: str, user: str, records) -> bool:
    # The same permission rule every retriever applies before ranking (Day 05): the user's groups from the directory
    return bool(set(records[cid].allowed_groups) & DIRECTORY.get(user, set()))


def expand_sections(ranked, user: str, question: str, setting: str = EXPANSION, records=None,
                    budget: int = BUDGET, hits: int = HITS) -> Context:
    # Section expansion: the hits first, then hit by hit the sections the setting adds around them. Skip a section
    # already sent, one the user may not read, and one that doesn't fit the budget (and try the next one).
    records = by_cid if records is None else records
    scores = dict(ranked)
    top = [cid for cid, _ in ranked[:hits]]
    candidates = [(cid, f"hit {k}") for k, cid in enumerate(top, start=1)] + \
                 [(x, f"{relation} of hit {k}") for k, cid in enumerate(top, start=1) for x, relation in around(cid, setting)]
    sent, steps, used = [], [], 0
    for cid, why in candidates:
        words = count_words(records[cid].text)
        if cid in sent:
            decision = "already sent"
        elif not may_read(cid, user, records):    # a lookup isn't a search: the search's filter doesn't cover it
            decision = "not readable"
        elif used + words > budget:
            decision = "over budget"
        else:
            decision = "sent"
            sent.append(cid)
            used += words
        score = scores.get(cid)
        steps.append(Step(cid, why, None if score is None else round(score, 4), words, decision))
    return Context(question, user, setting, tuple(sent), used, tuple(steps))


def context_for(q, setting: str = EXPANSION, records=None) -> Context:
    return expand_sections(rankings["weighted"][q.id], q.user, q.question, setting, records)


def show_trace(c: Context, labelled=()) -> None:
    print(f"{c.user}: {c.question}   [setting: {c.setting}, budget {BUDGET} words]")
    for s in c.trace:
        score = f"{s.score:.3f}" if s.score is not None else ""
        print(f"  {s.why:<20}{s.chunk:<56}{score:>7}{s.words:>5} words  {s.decision}"
              + ("   <- labelled" if s.chunk in labelled and s.decision == "sent" else ""))
    print(f"  sent {len(c.sent)} sections, {c.words} words")


show_trace(context_for(PHONE), PHONE.answer_chunks)          # the setting EXPANSION (section 0): what a pipeline sends
print()
show_trace(context_for(PHONE, "off"), PHONE.answer_chunks)
# The off-switch: "off" sends exactly the weighted fusion's top 3, for every query
off_is_top3 = all(context_for(q, "off").sent == tuple(cid for cid, _ in rankings["weighted"][q.id][:HITS])
                  for q in queries2)
print(f"\noff-switch: 'off' sends exactly the weighted fusion's top 3 for all {len(queries2)} queries: {off_is_top3}")

alice: My phone was stolen on a business trip. Who should I tell?   [setting: neighbours, budget 300 words]
  hit 1               travel-policy#safety-while-travelling                     1.000   42 words  sent
  hit 2               security-incident-reporting#after-you-report              0.654   37 words  sent
  hit 3               security-incident-reporting#what-not-to-do                0.568   38 words  sent
  previous of hit 1   travel-policy#claiming-travel-costs                       0.469   38 words  sent
  previous of hit 2   security-incident-reporting#what-not-to-do                0.568   38 words  already sent
  previous of hit 3   security-incident-reporting#how-to-report                 0.452   33 words  sent   <- labelled
  next of hit 3       security-incident-reporting#after-you-report              0.654   37 words  already sent
  sent 5 sections, 188 words

alice: My phone was stolen on a business trip. Who should I tell?   [setting: off, budget 300 words]
  hit 1   

The trace answers "why is this section in the context?" for every section: *How to report* is there as the section before hit 3, and the duplicates (hit 2's previous section is hit 3) are logged, not silently dropped. With the setting `off`, the same call sends exactly the weighted fusion's top 3, for every query: one switch takes the whole pattern out.

### 3.1 Which sections to add: the pre-registered rule, on dev

The rule, committed in [`evals/preregistration/day-11.md`](../evals/preregistration/day-11.md): *on dev's 90 answerable queries, the setting whose context holds a labelled chunk for the most queries; then the fewest words sent on average; then the earlier of previous, next, neighbours, parent.* Today dev is all of v2: v1's 81 queries and v2's 41 used test questions.

In [8]:
DEV = [q for q in queries3 if q.split == "dev" and q.answerable]   # the queries we may tune on: 90


def found(sent, q) -> bool:
    return any(cid in q.answer_chunks for cid in sent)


# Rule 1 (pre-registered; dev only): the most dev queries with a labelled chunk in the context, then the fewest words
# sent on average (to 0.1), then the earlier of previous, next, neighbours, parent
rule = {}
for s in ("off",) + SETTINGS:
    contexts = [context_for(q, s) for q in DEV]
    rule[s] = (sum(found(c.sent, q) for c, q in zip(contexts, DEV)), round(sum(c.words for c in contexts) / len(DEV), 1))
SETTING = max(SETTINGS, key=lambda s: (rule[s][0], -rule[s][1], -SETTINGS.index(s)))
print(f"{'dev only (90)':<14}{'labelled chunk in the context':>32}{'mean words sent':>18}")
for s, (n, words) in rule.items():
    print(f"{s:<14}{n:>32}{words:>18.1f}")
print(f"-> {SETTING}")
assert SETTING == PREREGISTERED["setting"] and rule == PREREGISTERED["dev"], \
    "the rule doesn't give the pre-registered setting or values: stop and find out why"

dev only (90)    labelled chunk in the context   mean words sent
off                                         88             115.6
previous                                    89             182.2
next                                        89             179.1
neighbours                                  90             223.1
parent                                      89             245.6
-> neighbours


`neighbours` wins by one query, the stolen phone, and sends about twice as many words as the top 3. `parent` sends the most words and finds one fewer: whole documents spend the budget on hit 1's document before reaching the others. The rule's choice comes from a question the pattern was aimed at, so every dev number here is **in-sample**.

## 4. On the same 122 queries: against the top 3, and against a control

Against the top 3 an expansion can't lose a question, since it sends the top 3 first. It also sends more text, and more text holds more answers whatever it is. So it is compared with a **control** too, which spends the same budget the simple way: for each query, the weighted fusion's top *m*, where *m* is the number of sections the expansion sent. If the expansion beats only the top 3, its gain came from sending more, not from the documents' structure.

The top 3 is read from `evals/baselines/day-10-retrieval-v2.json`, query by query, with the sign test. A difference is set apart as a near tie if the fused scores at the cut-off that decides it are within 0.001 (ranks 3 and 4, and *m* and *m* + 1 for the control).

In [9]:
STRATEGIES = ("top 3", f"weighted+{SETTING}", "control")
EXPANDED = f"weighted+{SETTING}"
CONTEXTS = {}      # (strategy, query id) -> the sections sent, in order


def add_contexts(qs) -> None:
    for q in qs:
        e = context_for(q, SETTING)
        CONTEXTS["top 3", q.id] = context_for(q, "off").sent
        CONTEXTS[EXPANDED, q.id] = e.sent
        # The control: as many sections as the expansion sent, taken from further down the ranking
        CONTEXTS["control", q.id] = tuple(cid for cid, _ in rankings["weighted"][q.id][:len(e.sent)])


def cutoff_gap(q, n: int) -> float:
    # How close the fused scores at ranks n and n + 1 are: under TIE, another CPU's rounding could swap them
    ranked = rankings["weighted"][q.id]
    return ranked[n - 1][1] - ranked[n][1] if n < len(ranked) else math.inf


def is_near(q, b: str) -> bool:
    # A difference is CPU rounding if the cut-off that decides it is a near tie: ranks 3 and 4 (which sections are the
    # hits), and, for the control, ranks m and m + 1
    gaps = [cutoff_gap(q, HITS)] + ([cutoff_gap(q, len(CONTEXTS[EXPANDED, q.id]))] if b == "control" else [])
    return min(gaps) < TIE


def words_of(strategy: str, q) -> int:
    return sum(count_words(by_cid[cid].text) for cid in CONTEXTS[strategy, q.id])


PAIRED, NEAR = {}, {}


def show_paired(b: str, parts, accepted=None) -> None:
    a = EXPANDED
    for part, selection in parts.items():
        def holds(strategy, q):
            if strategy == "top 3" and accepted is not None:     # the top 3 as recorded in the accepted baseline
                r = accepted["queries"][q.id]["weighted"]["first_relevant"]
                return r is not None and r <= HITS
            return found(CONTEXTS[strategy, q.id], q)
        has = {s: {q.id for q in selection if holds(s, q)} for s in (a, b)}
        near = {q.id for q in selection if q.id in has[a] ^ has[b] and is_near(q, b)}
        groups = {"both": has[a] & has[b], f"only {a}": has[a] - has[b] - near, f"only {b}": has[b] - has[a] - near,
                  "neither": {q.id for q in selection} - has[a] - has[b]}
        PAIRED[b, part] = {label: sorted(ids) for label, ids in groups.items()}
        NEAR[b, part] = sorted(near)
        p = sign_test(len(groups[f"only {a}"]), len(groups[f"only {b}"]))
        mean = {s: sum(words_of(s, q) for q in selection) / len(selection) for s in (a, b)}
        print(f"{a} vs {b}, {part} ({len(selection)}), labelled chunk in the context: "
              + " | ".join(f"{label} {len(ids)}" for label, ids in groups.items()) + f" | sign test p = {p:.2g}"
              + (f" | near ties set apart: {', '.join(sorted(near))}" if near else "")
              + f"\n    mean words sent: {a} {mean[a]:.0f}, {b} {mean[b]:.0f}")
        for label in (f"only {a}", f"only {b}"):
            if groups[label]:
                print(f"    {label:<30} " + ", ".join(sorted(groups[label])))


add_contexts(queries3[:122])
PARTS = {"v1 questions": [q for q in DEV if q.kind == "question" and not q.id.startswith("d10")],
         "lookups": [q for q in DEV if q.kind == "lookup"],
         "v2 questions (used)": [q for q in DEV if q.id.startswith("d10")]}
for b in ("top 3", "control"):
    show_paired(b, PARTS, accepted=day10)
    print()

weighted+neighbours vs top 3, v1 questions (31), labelled chunk in the context: both 31 | only weighted+neighbours 0 | only top 3 0 | neither 0 | sign test p = 1
    mean words sent: weighted+neighbours 207, top 3 111
weighted+neighbours vs top 3, lookups (30), labelled chunk in the context: both 30 | only weighted+neighbours 0 | only top 3 0 | neither 0 | sign test p = 1
    mean words sent: weighted+neighbours 254, top 3 118
weighted+neighbours vs top 3, v2 questions (used) (29), labelled chunk in the context: both 27 | only weighted+neighbours 2 | only top 3 0 | neither 0 | sign test p = 0.5
    mean words sent: weighted+neighbours 208, top 3 117
    only weighted+neighbours       d10-a12, d10-a24

weighted+neighbours vs control, v1 questions (31), labelled chunk in the context: both 31 | only weighted+neighbours 0 | only control 0 | neither 0 | sign test p = 1
    mean words sent: weighted+neighbours 207, control 210
weighted+neighbours vs control, lookups (30), labelled chunk in t

On dev the expansion adds two answers to the top 3, the lockout and the stolen phone, and one against the control, the stolen phone (*How to report* is 6th, the expansion sent 5 sections, and it is the section before hit 3). Both are questions the pattern was chosen for. Nothing else changes, at about twice the words.

## 5. The held-out test: 68 new questions

The comparison the pre-registration made: on the 52 answerable new questions, is a labelled chunk in the context of the top 3, of `weighted+neighbours`, and of the control? First, Day 10's five retrievers on the new questions, as a description.

In [10]:
add_rankings(NEW)          # the new questions: nothing about them was used to set anything today
add_contexts(NEW)
ANSWERABLE = [q for q in NEW if q.answerable]
HELD_OUT = {"v3 test questions": ANSWERABLE}
SELECTIONS_NEW = {"v3 test questions": ANSWERABLE}


def table(names, selections) -> None:
    width = max(len(f"{kind}, {name}") for kind in selections for name in names) + 2
    print(f"{'':<{width}}{'n':>3}" + "".join(f"{'hit@' + str(k):>7}" for k in K_VALUES) + f"{'MRR@10':>8}"
          "   hit@3 rate, 95% interval")
    for kind, selection in selections.items():
        for name in names:
            m = measure(rankings[name], selection)
            low, high = wilson(m["hit@3"], m["n"])
            print(f"{kind + ', ' + name:<{width}}{m['n']:>3}" + "".join(f"{m['hit@' + str(k)]:>7}" for k in K_VALUES)
                  + f"{m['MRR@10']:>8.2f}   {m['hit@3'] / m['n']:.0%} ({low:.0%} to {high:.0%})")


table(NAMES, SELECTIONS_NEW)
print()
print(f"{'':<40}{'n':>3}{'in context':>12}   rate, 95% interval{'mean words':>14}")
for s in STRATEGIES:
    n = sum(found(CONTEXTS[s, q.id], q) for q in ANSWERABLE)
    low, high = wilson(n, len(ANSWERABLE))
    print(f"v3 test questions, {s:<21}{len(ANSWERABLE):>3}{n:>12}   {f'{n / len(ANSWERABLE):.0%} ({low:.0%} to {high:.0%})':<19}"
          f"{sum(words_of(s, q) for q in ANSWERABLE) / len(ANSWERABLE):>11.0f}")
print()
for b in ("top 3", "control"):
    show_paired(b, HELD_OUT)

                                   n  hit@1  hit@3  hit@5 hit@10  MRR@10   hit@3 rate, 95% interval
v3 test questions, BM25           52     36     42     47     48    0.77   81% (68% to 89%)
v3 test questions, dense          52     38     48     52     52    0.84   92% (82% to 97%)
v3 test questions, weighted       52     40     49     52     52    0.86   94% (84% to 98%)
v3 test questions, BM25+map       52     35     42     48     48    0.77   81% (68% to 89%)
v3 test questions, weighted+map   52     40     49     52     52    0.86   94% (84% to 98%)

                                          n  in context   rate, 95% interval    mean words
v3 test questions, top 3                 52          49   94% (84% to 98%)           110
v3 test questions, weighted+neighbours   52          51   98% (90% to 100%)          199
v3 test questions, control               52          52   100% (93% to 100%)         199

weighted+neighbours vs top 3, v3 test questions (52), labelled chunk in the cont

The weighted fusion has 49 of the 52 answers in its top 3, and all 52 in its top 5. **The expansion reaches two of the three it misses** (the train-ticket and the suspicious-link questions): against the top 3 it gains two and loses none (p = 0.5). **The control reaches all three.** Against the control the expansion gains nothing and loses one, the taxi question.

### 5.1 Every miss

In [11]:
# Every v3 test question whose labelled chunk is missing from a context (written after reading each one).
# The assert fails if the set changes, so a new miss can't go unexplained.
CONTEXT_DIAGNOSES = {
    ("top 3", "d11-a06"): ("other section", "Scope, Travel and the travel policy's Booking travel come first; the "
                                            "procedure, Submitting a claim, is 5th"),
    ("top 3", "d11-a09"): ("other document", "three travel-policy sections come first; the expense policy's Travel "
                                             "section, which says ground transport is reimbursed, is 5th"),
    ("top 3", "d11-a40"): ("other section", "the event (What counts as an incident, What not to do) first; the "
                                            "procedure, How to report, 5th"),
    (EXPANDED, "d11-a09"): ("other document", "all three hits are in the travel policy, so their neighbours are too; "
                                              "the answer is in the expense policy (5th), which the control reaches"),
    ("control", "d10-a24"): ("other section", "How to report is 6th and the expansion sent 5 sections; it is the "
                                              "section before hit 3, so the expansion reaches it"),   # dev, v2 test (used)
}
missing = [(s, q) for s in STRATEGIES for q in ANSWERABLE if not found(CONTEXTS[s, q.id], q)]
assert {(s, q.id) for s, q in missing} == {k for k in CONTEXT_DIAGNOSES if k[1].startswith("d11")}, \
    "every miss needs a diagnosis, and only misses"
print(f"v3 test questions with the labelled chunk missing from the context: "
      + ", ".join(f"{s} {sum(x == s for x, _ in missing)}" for s in STRATEGIES))
for s, q in missing:
    r = first_relevant(rankings["weighted"][q.id], q.answer_chunks)
    category, why = CONTEXT_DIAGNOSES[s, q.id]
    print(f"  {s:<22}{q.id:<9}fused #{r or '-':<4}{category:<16}{q.question}\n{'':<45}{why}")

v3 test questions with the labelled chunk missing from the context: top 3 3, weighted+neighbours 1, control 0
  top 3                 d11-a06  fused #5   other section   I bought a train ticket for a client visit with my own card. How do I get the money back?
                                             Scope, Travel and the travel policy's Booking travel come first; the procedure, Submitting a claim, is 5th
  top 3                 d11-a09  fused #5   other document  Can I claim a taxi from the airport to my hotel?
                                             three travel-policy sections come first; the expense policy's Travel section, which says ground transport is reimbursed, is 5th
  top 3                 d11-a40  fused #5   other section   I clicked a link in a suspicious email. What should I do now?
                                             the event (What counts as an incident, What not to do) first; the procedure, How to report, 5th
  weighted+neighbours   d11-a09  fused #5  

All three misses have the stolen phone's shape: the question describes a situation and the answer is elsewhere. Twice it is in the same document as a hit: the train ticket's *Submitting a claim* follows the expense policy's *Scope* (hit 1), and the suspicious link's *How to report* follows *What counts as an incident* (hit 1). Once it is in **another document**: the taxi question's answer is the expense policy's *Travel* section, and all three hits are in the travel policy, so their neighbours are too. Neighbours can reach only the first kind. The ranking itself had put all three answers 5th, so reading a little further down reaches both kinds.

## 6. Permissions: a lookup is not a search

Day 05 showed that a metadata filter is access control only if nothing can skip it. Section expansion is a new way to skip it: the retriever filters what it **searches**, but neighbours are fetched **by position** from the document's structure, and the filter never sees them. Real documents often mix sections with different readers: an internal appendix, a draft, a managers-only note.

These documents set permissions per file, so on the real data no expansion can add a section its asker may not read (checked below for every context). To show the failure, one section is restricted to managers in an **in-memory copy** of the records (nothing on disk changes), the retrievers are rebuilt with that permission, and the expansion runs as alice.

In [12]:
# 1. Real data: no restricted chunk, and nothing the user may not read, in any context of any strategy
leaks = [(s, q.id) for (s, qid), sent in CONTEXTS.items() for q in [by_qid[qid]]
         if set(sent) & set(q.restricted_chunks) or not all(may_read(cid, q.user, by_cid) for cid in sent)]
print(f"contexts with a restricted or unreadable section: {len(leaks)} ({len(CONTEXTS)} contexts: "
      f"{len(STRATEGIES)} strategies x {len(queries3)} queries)")

# 2. A section-level restriction, on an in-memory copy of the records (the files don't change). Suppose the Security
# Team restricts one section to managers. Every retriever filters it, so the search never returns it to alice...
SECRET = "security-incident-reporting#after-you-report"
restricted = {**by_cid, SECRET: dataclasses.replace(by_cid[SECRET], allowed_groups=("managers",))}
bm25_alice = BM25Index({cid: c.text for cid, c in restricted.items() if may_read(cid, "alice", restricted)})


def weighted_restricted(q):
    lexical = [(h.id, h.score) for h in bm25_alice.search(q.question, len(chunks))]
    dense = [(cid, s) for cid, s in rankings["dense"][q.id] if may_read(cid, q.user, restricted)]  # exact search,
    return weighted(lexical, dense, ALPHA)                                                          # filtered first


# ... but its neighbours are still looked up by position in the document.
alice_queries = [q for q in queries3 if q.user == "alice"]
unchecked, checked = [], []
for q in alice_queries:
    ranked = weighted_restricted(q)
    top = [cid for cid, _ in ranked[:HITS]]
    assert SECRET not in top                                  # the search's filter works
    by_position = top + [x for cid in top for x, _ in around(cid, SETTING)]   # what an UNCHECKED lookup returns
    unchecked += [q.id] if SECRET in by_position else []
    checked += [q.id] if SECRET in expand_sections(ranked, q.user, q.question, SETTING, restricted).sent else []
print(f"\n{SECRET} restricted to managers (in memory): the search returns it to alice for 0 of her {len(alice_queries)} "
      f"queries; an unchecked lookup of the neighbours hands it to her for {len(unchecked)}; the checked expansion "
      f"for {len(checked)}\n")
show_trace(expand_sections(weighted_restricted(PHONE), PHONE.user, PHONE.question, SETTING, restricted), PHONE.answer_chunks)

contexts with a restricted or unreadable section: 0 (570 contexts: 3 strategies x 190 queries)

security-incident-reporting#after-you-report restricted to managers (in memory): the search returns it to alice for 0 of her 155 queries; an unchecked lookup of the neighbours hands it to her for 10; the checked expansion for 0

alice: My phone was stolen on a business trip. Who should I tell?   [setting: neighbours, budget 300 words]
  hit 1               travel-policy#safety-while-travelling                     1.000   42 words  sent
  hit 2               security-incident-reporting#what-not-to-do                0.568   38 words  sent
  hit 3               security-incident-reporting#what-counts-as-an-incident    0.499   45 words  sent
  previous of hit 1   travel-policy#claiming-travel-costs                       0.469   38 words  sent
  previous of hit 2   security-incident-reporting#how-to-report                 0.452   33 words  sent   <- labelled
  next of hit 2       security-inciden

The search never returns the restricted section to alice: every retriever filters it. But for 10 of her queries it sits next to one of her hits, so an unchecked lookup by position would hand it to her. The checked expansion skips it, and the trace says why ("not readable"). **Any step that fetches by id or by position (neighbours, a parent document, a linked document) needs the same permission check as the search**, and a test that proves it.

## 7. What it costs

In [13]:
def tokens(strategy: str, q) -> int:
    # The embedding model's own tokenizer (the same tokenizer.json as Day 09's reranker): what a model would read
    return sum(len(embedder.tokenizer.encode(by_cid[cid].text, add_special_tokens=False).ids)
               for cid in CONTEXTS[strategy, q.id])


GROUPS = {"answerable (dev + test)": [q for q in queries3 if q.answerable],
          "unanswerable questions": [q for q in queries3 if q.kind == "question" and not q.answerable]}
COST = {}
print(f"{'':<34}" + "".join(f"{s:>22}" for s in STRATEGIES))
for label, qs in GROUPS.items():
    for unit, count in (("words", words_of), ("tokens", tokens)):
        row = [(sum(count(s, q) for q in qs) / len(qs), max(count(s, q) for q in qs)) for s in STRATEGIES]
        COST[label, unit] = row
        print(f"{label + ', ' + unit:<34}" + "".join(f"{f'mean {m:.0f}, max {x}':>22}" for m, x in row))


def timed(work, sample, rounds: int = 3) -> tuple[float, float]:
    samples = []
    for _ in range(rounds):
        for q in sample:
            started = time.perf_counter()
            work(q)
            samples.append((time.perf_counter() - started) * 1000)
    samples.sort()
    return statistics.median(samples), samples[max(0, math.ceil(0.95 * len(samples)) - 1)]


over_budget = [q.id for q in queries3 if words_of("control", q) > BUDGET]
print(f"\ncontrol contexts over the {BUDGET}-word budget: {len(over_budget)} of {len(queries3)} "
      f"({sum(by_qid[qid].kind == 'lookup' for qid in over_budget)} of them lookups)")
SAMPLE = queries3[::6]     # 32 queries from every split
timings = {"section expansion (lookup + trace)": timed(lambda q: context_for(q, SETTING), SAMPLE),
           "dense (embed the question + search)": timed(lambda q: dense_search(q.user, q.question), SAMPLE)}
print()
for step, (median, p95) in timings.items():
    print(f"{step:<38} median {median:9.3f} ms   p95 {p95:9.3f} ms")

                                                   top 3   weighted+neighbours               control
answerable (dev + test), words         mean 114, max 165     mean 214, max 300     mean 217, max 346


answerable (dev + test), tokens        mean 137, max 195     mean 259, max 378     mean 262, max 426
unanswerable questions, words          mean 107, max 142     mean 217, max 297     mean 213, max 308
unanswerable questions, tokens         mean 129, max 164     mean 264, max 369     mean 258, max 363

control contexts over the 300-word budget: 9 of 190 (6 of them lookups)



section expansion (lookup + trace)     median     0.022 ms   p95     0.037 ms
dense (embed the question + search)    median    12.187 ms   p95    19.559 ms


Section expansion sends about twice the words and tokens of the top 3, on every kind of question, unanswerable ones included; the control sends about as many (that is its point). The expansion step itself takes hundredths of a millisecond. The cost is in what the reader reads: twice the tokens for a hosted model, and twice as much text for any reader to misuse. The expansion's largest context hits the 300-word budget exactly. The control has no budget of its own and goes over it for 9 of the 190 queries (6 of them lookups); of the three questions among them, two are refused at the gate below, and Day 06's `assemble()` trims the third before the reader.

## 8. End to end: does the reader use the extra sections?

A description, as pre-registered: Day 06's steps with Day 07's measured settings, fed each of the three contexts. The gate reads dense retrieval's top score (0.62; a fused score is near 1 for every question by construction), then the extractive reader at 0.63 with Day 06's names rule, then the citation check. The reader embeds every sentence of every source it gets, and the three contexts share most sources, so a cache keeps each sentence's vector by its text; the cached reader is checked against Day 06's on a sample first.

In [14]:
class SentenceCache:
    """The reader embeds every sentence of every source it gets, and the three contexts share most sources. This keeps
    each sentence's vector by its exact text, so it is embedded once (alone or in a batch gives the same vector)."""

    def __init__(self, embedder):
        self.embedder, self.vectors = embedder, {}
        self.model = embedder.model                       # what the reader may ask about the model

    def embed(self, texts):
        new = [t for t in dict.fromkeys(texts) if t not in self.vectors]
        if new:
            self.vectors.update(zip(new, self.embedder.embed(new)))
        return np.stack([self.vectors[t] for t in texts])

    def embed_query(self, question, *args, **kwargs):
        return self.embedder.embed_query(question, *args, **kwargs)


def answer_with(q, sent, reader) -> RagAnswer:
    # Day 06's answer() with the context chosen here: Day 07's gate on the dense top score (a fused score is near 1 by
    # construction), the permission check again before sending, the budget, the reader, the citation check
    groups = DIRECTORY.get(q.user, set())
    if any(not set(by_cid[cid].allowed_groups) & groups for cid in sent):
        raise PermissionError(f"a context holds a chunk {q.user} may not read")
    if rankings["dense"][q.id][0][1] < DENSE_GATE:
        return RagAnswer(q.user, q.question, "abstained", ABSTAIN, "below the dense gate: no generator called")
    scores = dict(rankings["weighted"][q.id])
    context = assemble(q.question, [Result(i, scores[cid], cid, by_cid[cid]) for i, cid in enumerate(sent, 1)], BUDGET)
    reply = reader.generate(context)
    check = check_citations(reply.text, context)
    status = "abstained" if check.abstained else "blocked" if not check.passed else "answered"
    return RagAnswer(q.user, q.question, status, {"abstained": ABSTAIN, "blocked": BLOCKED}.get(status, reply.text),
                     reply.note, context, check, reply)


natural = [q for q in queries3 if q.kind == "question"]
reader = ExtractiveReader(SentenceCache(embedder), min_score=READER_THRESHOLD)
plain = ExtractiveReader(embedder, min_score=READER_THRESHOLD)
same_reader = all(answer_with(q, CONTEXTS[EXPANDED, q.id], reader) == answer_with(q, CONTEXTS[EXPANDED, q.id], plain)
                  for q in natural[::5])
print(f"the cached reader answers exactly as Day 06's reader (every 5th question, {len(natural[::5])}): {same_reader}")
answers = {s: {q.id: answer_with(q, CONTEXTS[s, q.id], reader) for q in natural} for s in STRATEGIES}
trimmed = [q.id for q in natural if len(answers["control"][q.id].context.sources if answers["control"][q.id].context
                                         else CONTEXTS["control", q.id]) < len(CONTEXTS["control", q.id])]
outcomes = {s: {q.id: outcome(answers[s][q.id], q) for q in natural} for s in STRATEGIES}
E2E_SPLITS = {"dev": [q for q in natural if q.split == "dev"], "v3 test": [q for q in natural if q.split == "test"]}
E2E = {}
print(f"\n{'gate 0.62, reader 0.63':<34}{'answerable: correct / wrong / refused':<40}not answerable: refused / answered")
for s in STRATEGIES:
    for split, part in E2E_SPLITS.items():
        a = Counter(outcomes[s][q.id] for q in part if q.answerable)
        u = Counter(outcomes[s][q.id] for q in part if not q.answerable)
        E2E[s, split] = (a["correct"], a["wrong"], a["refused"], u["refused"], u["answered"])
        answerable = f"{a['correct']} / {a['wrong']} / {a['refused']}  (of {sum(a.values())})"
        print(f"{s + ', ' + split:<34}{answerable:<40}{u['refused']} / {u['answered']}  (of {sum(u.values())})")
CHANGED = [q for q in natural if len({outcomes[s][q.id] for s in STRATEGIES}) > 1]
ADDED = [q for q in natural if q.answerable and found(CONTEXTS[EXPANDED, q.id], q) and not found(CONTEXTS["top 3", q.id], q)]
print(f"\nwhere the expansion put the labelled chunk in the context and the top 3 didn't ({len(ADDED)}):")
for q in ADDED:
    a = answers[EXPANDED][q.id]
    print(f"  {q.id:<9}{q.question}\n{'':<11}{outcomes['top 3'][q.id]} -> {outcomes[EXPANDED][q.id]}: {a.text[:118]}")
print(f"\ncontrol contexts trimmed by Day 06's budget before the reader: {len(trimmed)}")
print(f"questions whose outcome changes between contexts ({len(CHANGED)}):")
for q in CHANGED:
    print(f"  {q.id:<9}{q.question}\n{'':<11}" + " | ".join(f"{s}: {outcomes[s][q.id]}" for s in STRATEGIES))
    for s in STRATEGIES:
        print(f"{'':<11}{s}: {answers[s][q.id].text[:160]}")


def why_refused(a: RagAnswer) -> str:
    if a.status == "blocked":
        return "citation check"
    return ("gate" if a.reason.startswith("below the dense gate") else "names rule" if a.reason.startswith("no source "
            "mentions") else "reader threshold" if a.reason.startswith("best sentence") else a.reason)


REFUSED = {s: Counter(why_refused(answers[s][q.id]) for q in E2E_SPLITS["v3 test"]
                      if q.answerable and outcomes[s][q.id] == "refused") for s in STRATEGIES}
first_word = {q.id for q in E2E_SPLITS["v3 test"] if q.answerable and why_refused(answers["top 3"][q.id]) == "names rule"
              and set(answers["top 3"][q.id].reason.removeprefix("no source mentions ").split(", "))
              <= {s.split()[0] for s in SENTENCE.split(q.question)[1:]}}
print("\nwhy answerable v3 test questions were refused: " + " | ".join(
    f"{s}: " + ", ".join(f"{why} {n}" for why, n in REFUSED[s].most_common()) for s in STRATEGIES))
print(f"of the top 3's {REFUSED['top 3']['names rule']} names-rule refusals, {len(first_word)} name only the first word "
      "of the question's second sentence (Day 06's rule skips only the first word of the question)")

the cached reader answers exactly as Day 06's reader (every 5th question, 32): True



gate 0.62, reader 0.63            answerable: correct / wrong / refused   not answerable: refused / answered
top 3, dev                        45 / 6 / 9  (of 60)                     21 / 11  (of 32)
top 3, v3 test                    24 / 6 / 22  (of 52)                    8 / 8  (of 16)
weighted+neighbours, dev          45 / 6 / 9  (of 60)                     21 / 11  (of 32)
weighted+neighbours, v3 test      25 / 7 / 20  (of 52)                    8 / 8  (of 16)
control, dev                      45 / 6 / 9  (of 60)                     21 / 11  (of 32)
control, v3 test                  25 / 7 / 20  (of 52)                    8 / 8  (of 16)

where the expansion put the labelled chunk in the context and the top 3 didn't (4):
  d10-a12  What happens if I type my password wrong too many times?
           wrong -> wrong: Passwords must be at least 14 characters long [password-and-mfa-standard#passwords]. Never reuse your Acme password on
  d10-a24  My phone was stolen on a business trip. 

**The four questions where the expansion put the answer in the context don't get it right.** The lockout question still gets the password-length sentence. The suspicious-link question now has *How to report* in its context, and the reader still copies "Do not try to investigate the incident yourself…" from the section next to it. The stolen phone is still refused for "Who". The train ticket goes from refused to wrong.

**On dev no outcome changes; on the new questions three change, and none because of a section the expansion added.** All three (the train ticket, the raise question, the desk allowance) were refused under the top 3 by Day 06's names rule, which takes the first word of a question's second sentence ("How", "Can") for a name. A bigger context happens to contain that word somewhere ("This policy explains how…", "Leave can be taken…"), so the rule lets the question through: one becomes correct with the expansion, one correct only with the control's context, and the train ticket wrong with both.

Two lessons:

- **Evidence in the context is necessary, not sufficient.** Day 06's reader copies the sentences closest to the question, and a question that describes an event is closest to the section that describes the event. Day 12 evaluates answers.
- **The names rule is the biggest single cause of refusals here**: 18 of the 22 refused answerable new questions, 16 of them for the second sentence's first word, because 24 of the 68 new questions have two sentences. Day 10 measured a fix and didn't adopt it (it let one more unanswerable question through); it is worth revisiting with Day 12's answer evaluation.

## 9. Decision

The decision, and what it needed, were written down before any held-out question was scored ([`evals/preregistration/day-11.md`](../evals/preregistration/day-11.md)):

In [15]:
gained = sorted(PAIRED["control", "v3 test questions"][f"only {EXPANDED}"])
lost = sorted(PAIRED["control", "v3 test questions"]["only control"])
D1 = bool(gained) and not lost
print(f"D1, section expansion in the default context: on the {len(ANSWERABLE)} answerable v3 test questions, "
      f"{EXPANDED}'s context holds a labelled chunk the control's doesn't for {', '.join(gained) or 'none'}, and the "
      f"control's one {EXPANDED}'s doesn't for {', '.join(lost) or 'none'}"
      + (f" (near ties set apart: {', '.join(NEAR['control', 'v3 test questions'])})" if NEAR["control", "v3 test questions"] else "")
      + " -> " + ("adopted: later days send the expanded context" if D1 else
                  "not adopted: the top 3 stays; the expansion is recorded as measured"))

D1, section expansion in the default context: on the 52 answerable v3 test questions, weighted+neighbours's context holds a labelled chunk the control's doesn't for none, and the control's one weighted+neighbours's doesn't for d11-a09 -> not adopted: the top 3 stays; the expansion is recorded as measured


**Not adopted.** On questions it wasn't written for, reading around the hit found two answers the top 3 missed; reading further down the ranking, with the same number of sections, found those two and one more. On this corpus the weighted fusion ranks every new question's answer 5th or better, so the documents' structure adds nothing that depth doesn't. The top 3 stays the default context, and the expansion is recorded with its numbers.

That isn't a verdict on the pattern everywhere. Section expansion earns its place when chunks are smaller than the unit of meaning (sentences, or Day 03's fixed windows), when answers often span neighbouring chunks (a procedure split across chunks), or when the ranking can't reach the right section but does reach its neighbour. Here the chunks are whole sections and the hybrid ranks well. **The control is the reusable lesson**: whenever a change adds context, compare it with the same amount of context chosen the simple way.

The new test split is used now: it decided D1.

## 10. Save the run, compare with the accepted baseline

One run file for v3 (190 queries: Day 10's five retrievers, and `weighted+neighbours`, whose entry records the sections it sent), and the traces of every query, one JSON line each. The accepted copy of the run is committed in [`evals/baselines/`](../evals/baselines/) as `day-11-retrieval-v3.json`, the reference for later days.

In [16]:
DIAGNOSES = {(e["retriever"], e["id"]): (e["category"], e["why"]) for e in day10["errors"]}   # Day 10's, on v2
NEW_DIAGNOSES = {   # every v3 test miss of Day 10's five retrievers (written after reading each one)
    ("BM25", "d11-a06"): ("vocabulary", "no word in common with either answer: bought, ticket, card, money back vs submit, "
                                        "claims, expense system"),
    ("BM25", "d11-a07"): ("vocabulary", "only 'claim' is shared; four and months pull the parental-leave sections up "
                                        "(four weeks, 12 months)"),
    ("BM25", "d11-a08"): ("vocabulary", "get the money vs paid with the next monthly payroll; Scope, Equipment and "
                                        "Submitting a claim share more of its words"),
    ("BM25", "d11-a09"): ("vocabulary", "no word in common: taxi vs ground transport, trains, buses and rideshares"),
    ("BM25", "d11-a10"): ("vocabulary", "download and web vs install and internet; the VPN guide's 'do not download "
                                        "VPN software' comes first"),
    ("BM25", "d11-a15"): ("vocabulary", "raise and a number vs a specific increase; only 'team' is shared"),
    ("BM25", "d11-a32"): ("header words", "every remote-work section carries 'Remote Work Policy' in its header, so "
                                          "they tie on the question's words; contractor vs Contractors (no stemming)"),
    ("BM25", "d11-a36"): ("vocabulary", "start at 7, log off at 2 vs between 10:00 and 15:00; only 'work' is shared"),
    ("BM25", "d11-a40"): ("other section", "What counts as an incident has 'clicking a suspicious link'; How to report "
                                           "shares no word with the question"),
    ("BM25", "d11-a47"): ("other section", "the limit is in Daily limits by city, what to do above it in Claiming "
                                           "travel costs (4th)"),
    ("dense", "d11-a06"): ("other section", "the expense policy's Travel section (trains) comes first; the procedure, "
                                            "Submitting a claim, is 4th"),
    ("dense", "d11-a40"): ("other section", "the event (What not to do, What counts as an incident) outranks the "
                                            "procedure (How to report, 5th)"),
    ("dense", "d11-a43"): ("buried sentence", "regulators are in the second of two sentences; How to report and the "
                                              "retention policy's Purpose score higher (4th)"),
    ("dense", "d11-a47"): ("other section", "the table and Booking travel come first; Claiming travel costs is 4th"),
    ("weighted", "d11-a06"): CONTEXT_DIAGNOSES["top 3", "d11-a06"],
    ("weighted", "d11-a09"): CONTEXT_DIAGNOSES["top 3", "d11-a09"],
    ("weighted", "d11-a40"): CONTEXT_DIAGNOSES["top 3", "d11-a40"],
    ("BM25+map", "d11-a36"): ("vocabulary", "the map adds remotely and remote, which every remote-work section has: "
                                            "Core hours moves from 9th to 5th"),
}
for (base, mapped) in (("BM25", "BM25+map"), ("weighted", "weighted+map")):   # where the map met nothing
    for (name, qid), (category, _) in list(NEW_DIAGNOSES.items()):
        if name == base and (mapped, qid) not in NEW_DIAGNOSES and not expand(by_qid[qid].question, ENTRIES).matched:
            NEW_DIAGNOSES[mapped, qid] = (category, f"as for {'BM25' if base == 'BM25' else 'the weighted fusion'}: "
                                                    "the map added nothing")
DIAGNOSES.update(NEW_DIAGNOSES)
misses = [(name, q) for name in NAMES for q in queries3 if q.answerable and not hit(rankings[name][q.id], q.answer_chunks, HITS)]
assert {(name, q.id) for name, q in misses if q.split == "test"} == set(NEW_DIAGNOSES) and \
    all((name, q.id) in DIAGNOSES for name, q in misses), "every miss needs a diagnosis, and only misses"
context_misses = [(s, q) for s in (EXPANDED, "control") for q in queries3 if q.answerable and not found(CONTEXTS[s, q.id], q)]
order = {(e["retriever"], e["id"]): i for i, e in enumerate(day10["errors"])}     # Day 10's errors first, in its order,
ordered_misses = sorted([(n, q) for n, q in misses if q.split == "dev"], key=lambda m: order[m[0], m[1].id]) + \
    [(n, q) for n, q in misses if q.split == "test"]                              # then today's new questions
assert {(s, q.id) for s, q in context_misses} == {k for k in CONTEXT_DIAGNOSES if k[0] != "top 3"}
print("v3 test, misses in the top 3: " + ", ".join(f"{name} {sum(n == name and q.split == 'test' for n, q in misses)}"
                                                    for name in NAMES))

EXPANSION_SETTINGS = {
    "code": "expand_sections() in the Day 11 notebook", "expands": "weighted", "hits": HITS, "setting": SETTING,
    "setting_rule": "most answerable dev queries with a labelled chunk in the context, then the fewest words sent on "
                    "average, then previous, next, neighbours, parent",
    "order": "the hits in rank order, then hit by hit the section just before and the one just after it in its document",
    "budget_words": BUDGET, "skips": "a section already sent, one the user may not read, one over the budget",
    "permissions": "every added section is checked against the user's groups (a lookup is not a search)",
    "control": "the weighted fusion's top m, m = the number of sections sent",
    "first_relevant": "the position of the first labelled chunk among the sections sent, or None if none was sent",
    "margin": "the fused score gap between ranks 3 and 4: under 0.001, the hits themselves are a near tie"}


def expansion_entry(q) -> dict:
    sent, m = CONTEXTS[EXPANDED, q.id], len(CONTEXTS[EXPANDED, q.id])
    gap, control_gap = cutoff_gap(q, HITS), cutoff_gap(q, m)
    return {"first_relevant": next((i for i, cid in enumerate(sent, 1) if cid in q.answer_chunks), None) if q.answerable else None,
            "margin": round(gap, 4) if q.answerable and gap != math.inf else None,
            "top3": list(sent[:HITS]), "top_score": top_score(rankings["weighted"][q.id]),
            "sent": list(sent), "words": words_of(EXPANDED, q),
            "control_found": found(CONTEXTS["control", q.id], q) if q.answerable else None,
            "control_margin": round(control_gap, 4) if q.answerable and control_gap != math.inf else None}


def context_metrics(selection) -> dict:
    return {"n": len(selection), "top 3 in context": sum(found(CONTEXTS["top 3", q.id], q) for q in selection),
            "in context": sum(found(CONTEXTS[EXPANDED, q.id], q) for q in selection),
            "control in context": sum(found(CONTEXTS["control", q.id], q) for q in selection),
            "mean words": round(sum(words_of(EXPANDED, q) for q in selection) / len(selection), 1),
            "control mean words": round(sum(words_of("control", q) for q in selection) / len(selection), 1)}


DATASETS[3] = {"questions": V3["questions"].relative_to(REPO_ROOT).as_posix(), "version": manifest3["version"],
               "sha256": v3_sha256, "snapshot": manifest3["snapshot"]["fingerprint"]}
SELECTIONS3 = {"questions": [q for q in queries3 if q.kind == "question" and q.answerable],
               "lookups": [q for q in queries3 if q.kind == "lookup"]}
run = {"name": "Day 11 retrieval baseline (questions v3)", "dataset": DATASETS[3],
       "retrievers": {**{name: SETTINGS_OF[name] for name in NAMES}, EXPANDED: EXPANSION_SETTINGS}, "top_k": HITS,
       "metrics": {**metrics_of(NAMES, SELECTIONS3),
                   **{f"{kind}, {EXPANDED}": context_metrics(sel) for kind, sel in SELECTIONS3.items()},
                   f"v3 test questions, {EXPANDED}": context_metrics(ANSWERABLE)},
       "queries": {q.id: {**{name: entry(name, q) for name in NAMES}, EXPANDED: expansion_entry(q)} for q in queries3},
       "errors": [{"retriever": name, "id": q.id, "first_relevant": first_relevant(rankings[name][q.id], q.answer_chunks),
                   "category": DIAGNOSES[name, q.id][0], "why": DIAGNOSES[name, q.id][1]} for name, q in ordered_misses]
                 + [{"retriever": EXPANDED if s == EXPANDED else "control (weighted top m)", "id": q.id,
                     "first_relevant": first_relevant(rankings["weighted"][q.id], q.answer_chunks),
                     "category": CONTEXT_DIAGNOSES[s, q.id][0], "why": CONTEXT_DIAGNOSES[s, q.id][1]}
                    for s, q in context_misses],
       "thresholds": {"preregistration": PREREGISTRATION.relative_to(REPO_ROOT).as_posix(),
                      "setting_rule_on_dev": {s: {"found": n, "mean_words": w} for s, (n, w) in rule.items()},
                      "permissions_demo": {"restricted": SECRET, "alice's queries": len(alice_queries),
                                           "unchecked lookup sends it": len(unchecked), "checked expansion sends it": len(checked)},
                      "day10": "see evals/baselines/day-10-retrieval-v2.json (the map and the new names rule, not adopted)",
                      "decisions": {"D1 section expansion in the default context": "adopted" if D1 else "not adopted"}}}
OUT_DIR.mkdir(parents=True, exist_ok=True)
run_file = OUT_DIR / BASELINE_FILE.name
run_file.write_text(json.dumps(run, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
traces_file = OUT_DIR / "traces-v3.jsonl"     # one line per query: every section considered, and what happened to it
traces_file.write_text("".join(json.dumps({"id": q.id, **dataclasses.asdict(context_for(q))}, ensure_ascii=False)
                               + "\n" for q in queries3), encoding="utf-8", newline="\n")
print(f"this run: {run_file.relative_to(REPO_ROOT).as_posix()} ({len(queries3)} queries x {len(run['retrievers'])} "
      f"retrievers, {len(run['errors'])} errors) and {traces_file.relative_to(REPO_ROOT).as_posix()} ({len(queries3)} traces)")
if BASELINE_FILE.is_file():
    changes, ties = compare(run, json.loads(BASELINE_FILE.read_text(encoding="utf-8")))
    print(f"compared with {BASELINE_FILE.relative_to(REPO_ROOT).as_posix()}: "
          + (f"{len(changes)} changes" + "".join(f"\n  {c}" for c in changes) if changes else
             f"no changes ({len(queries3)} queries x {len(run['retrievers'])} retrievers and their metrics)"))
    for t in ties:
        print(f"  near tie, not a change: {t}")
else:
    changes, ties = None, []
    print(f"no accepted baseline yet: review {run_file.name}, then copy it to "
          f"{BASELINE_FILE.relative_to(REPO_ROOT).as_posix()} and commit it")
closest = sorted((min(e["margin"], e["control_margin"] or math.inf), qid) for qid, per in run["queries"].items()
                 for e in [per[EXPANDED]] if e["margin"] is not None)[:3]
print("closest cut-offs for the expansion and the control: " + ", ".join(f"{qid} ({m})" for m, qid in closest))

v3 test, misses in the top 3: BM25 10, dense 4, weighted 3, BM25+map 10, weighted+map 3
this run: outputs/day-11/day-11-retrieval-v3.json (190 queries x 6 retrievers, 59 errors) and outputs/day-11/traces-v3.jsonl (190 traces)
compared with evals/baselines/day-11-retrieval-v3.json: no changes (190 queries x 6 retrievers and their metrics)
closest cut-offs for the expansion and the control: d03-03 (0.0002), d11-a28 (0.0003), d07-a03 (0.0005)


The closest cut-offs are under 0.001, so another CPU could swap the two sections there, but they decide nothing: in each of those queries the labelled chunk is ranked above the cut-off (a sanity check below asserts this for every query).

## 11. Sanity checks

Each check pins a result quoted in this notebook or in LEARN.md. Ids, ranks and counts are compared exactly; timings only for their shape.

In [17]:
def counts(name: str, selection) -> tuple:
    m = measure(rankings[name], selection)
    return m["hit@1"], m["hit@3"], m["hit@5"], m["hit@10"]


def n_found(strategy: str, selection) -> int:
    return sum(found(CONTEXTS[strategy, q.id], q) for q in selection)


# The same inputs as Day 10 (section 1), and the new questions
assert problems2 == [] and changes10 == [] and same_top3 and len(queries2) == 122 and len(chunks) == 44
assert problems3 == [] and same122 and len(queries3) == 190 and len(NEW) == 68 and len(ANSWERABLE) == 52
assert manifest3["counts"]["dev"] == 122 and manifest3["counts"]["test"] == 68 and manifest3["previous"]["version"] == 2

# The stolen phone (section 2) and the expansion (section 3)
assert first_relevant(rankings["weighted"][PHONE.id], PHONE.answer_chunks) == 6 and where(PHONE.answer_chunks[0]) == "section 2 of 4"
assert first_relevant(rankings["weighted"][LOCKOUT.id], LOCKOUT.answer_chunks) == 5
phone = context_for(PHONE, SETTING)
assert [s.decision for s in phone.trace] == ["sent"] * 4 + ["already sent", "sent", "already sent"] and phone.words == 188
assert [s.why for s in phone.trace if s.chunk in PHONE.answer_chunks] == ["previous of hit 3"]
assert all(context_for(q, "off").sent == CONTEXTS["top 3", q.id] == tuple(cid for cid, _ in rankings["weighted"][q.id][:HITS])
           for q in queries3)                                                   # the off-switch, for every query
assert all(CONTEXTS[EXPANDED, q.id][:HITS] == CONTEXTS["top 3", q.id] and words_of(EXPANDED, q) <= BUDGET and
           set(CONTEXTS[EXPANDED, q.id][HITS:]) <= {x for h in CONTEXTS["top 3", q.id] for x, _ in around(h, SETTING)}
           for q in queries3)                       # hits first, then only their neighbours, never over the budget
assert around("security-incident-reporting#what-not-to-do", "neighbours") == [
    ("security-incident-reporting#how-to-report", "previous"), ("security-incident-reporting#after-you-report", "next")]
assert around("travel-policy#safety-while-travelling", "next") == [] and len(around("travel-policy#purpose", "parent")) == 4
assert rule == PREREGISTERED["dev"] and SETTING == "neighbours"

# Dev, against the top 3 and the control (section 4)
assert PAIRED["top 3", "v2 questions (used)"][f"only {EXPANDED}"] == ["d10-a12", "d10-a24"]
assert PAIRED["control", "v2 questions (used)"][f"only {EXPANDED}"] == ["d10-a24"]
assert all(not PAIRED[b, part][f"only {b}"] for b in ("top 3", "control") for part in PARTS)          # no loss on dev
assert all(not PAIRED[b, part][f"only {EXPANDED}"] for b in ("top 3", "control") for part in ("v1 questions", "lookups"))

# The held-out questions (section 5)
assert counts("weighted", ANSWERABLE) == counts("weighted+map", ANSWERABLE) == (40, 49, 52, 52)
assert counts("dense", ANSWERABLE) == (38, 48, 52, 52) and counts("BM25", ANSWERABLE) == (36, 42, 47, 48)
assert counts("BM25+map", ANSWERABLE) == (35, 42, 48, 48)
assert [n_found(s, ANSWERABLE) for s in STRATEGIES] == [49, 51, 52]
assert PAIRED["top 3", "v3 test questions"][f"only {EXPANDED}"] == ["d11-a06", "d11-a40"]
assert PAIRED["top 3", "v3 test questions"]["only top 3"] == [] and PAIRED["control", "v3 test questions"][f"only {EXPANDED}"] == []
assert PAIRED["control", "v3 test questions"]["only control"] == ["d11-a09"] and all(not ids for ids in NEAR.values())
assert {q.id for s, q in missing} == {"d11-a06", "d11-a09", "d11-a40"} and \
    all(first_relevant(rankings["weighted"][qid], by_qid[qid].answer_chunks) == 5 for qid in ("d11-a06", "d11-a09", "d11-a40"))
# No near tie decides anything: where a cut-off is within TIE, the labelled chunk is ranked before it on both sides
for q in queries3:
    cuts = [n for n in (HITS, len(CONTEXTS[EXPANDED, q.id])) if cutoff_gap(q, n) < TIE]
    assert not q.answerable or not cuts or first_relevant(rankings["weighted"][q.id], q.answer_chunks) < min(cuts), q.id

# Permissions (section 6)
assert leaks == [] and len(unchecked) == 10 and checked == [] and len(alice_queries) == 155

# Cost (section 7): the shape only for timings, with a wide margin
assert all(COST[g, "words"][1][1] <= BUDGET for g in GROUPS) and COST["answerable (dev + test)", "words"][2][1] > BUDGET
assert len(over_budget) == 9 and [qid for qid in over_budget if by_qid[qid].kind == "question"] == ["d07-u05", "d10-a28", "d11-u08"]
assert all(COST[g, unit][1][0] > 1.7 * COST[g, unit][0][0] for g in GROUPS for unit in ("words", "tokens"))
assert timings["section expansion (lookup + trace)"][0] < timings["dense (embed the question + search)"][0] / 50

# End to end (section 8)
assert same_reader and [q.id for q in CHANGED] == ["d11-a06", "d11-a15", "d11-a38"] and trimmed == ["d07-u05"]
assert all(E2E[s, "dev"] == (45, 6, 9, 21, 11) for s in STRATEGIES)
assert E2E["top 3", "v3 test"] == (24, 6, 22, 8, 8) and E2E[EXPANDED, "v3 test"] == E2E["control", "v3 test"] == (25, 7, 20, 8, 8)
assert [outcomes[s]["d11-a06"] for s in STRATEGIES] == ["refused", "wrong", "wrong"]
assert [outcomes[s]["d11-a15"] for s in STRATEGIES] == ["refused", "correct", "refused"]
assert [outcomes[s]["d11-a38"] for s in STRATEGIES] == ["refused", "refused", "correct"]
assert all(answers["top 3"][qid].reason.startswith("no source mentions") for qid in ("d11-a06", "d11-a15", "d11-a38"))
assert [q.id for q in ADDED] == ["d10-a12", "d10-a24", "d11-a06", "d11-a40"]
assert all(outcomes[s]["d11-a40"] == "wrong" and outcomes[s]["d10-a12"] == "wrong" and outcomes[s]["d10-a24"] == "refused"
           for s in STRATEGIES)                               # where the expansion added the answer, the outcome stays
assert answers[EXPANDED]["d11-a40"].text.startswith("Do not try to investigate the incident yourself")
assert REFUSED["top 3"]["names rule"] == 18 and len(first_word) == 16 and sum(REFUSED["top 3"].values()) == 22

# Decision (section 9) and the run (section 10)
assert not D1 and gained == [] and lost == ["d11-a09"]
saved = json.loads(run_file.read_text(encoding="utf-8"))
assert saved["dataset"]["version"] == 3 and len(saved["errors"]) == 59 and len(saved["queries"]) == 190
assert saved["errors"][:len(day10["errors"])] == day10["errors"]                    # Day 10's errors, unchanged
assert all(saved["queries"][q.id][n] == reproduced["queries"][q.id][n] for q in queries2 for n in NAMES)   # = Day 10's
assert len(traces_file.read_text(encoding="utf-8").splitlines()) == 190
assert changes is None or changes == [], changes            # near ties (CPU rounding) are allowed
if BASELINE_FILE.is_file():                                 # and the same top 3 and sections sent as the accepted run,
    accepted = json.loads(BASELINE_FILE.read_text(encoding="utf-8"))    # except where a cut-off is a near tie
    steady = {q.id for q in queries3 if min(cutoff_gap(q, HITS), cutoff_gap(q, len(CONTEXTS[EXPANDED, q.id]))) >= TIE}
    assert all(saved["queries"][qid][n]["top3"] == accepted["queries"][qid][n]["top3"]
               for qid in steady for n in accepted["retrievers"])
    assert all(saved["queries"][qid][EXPANDED]["sent"] == accepted["queries"][qid][EXPANDED]["sent"] for qid in steady)

print("All Day 11 sanity checks passed.")

All Day 11 sanity checks passed.


## 12. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing anything. Today's test split has been used to decide, so treat everything here as dev.

1. **Flip the switch.** Set `EXPANSION = "off"` in section 0 and rerun. What changes in section 3 and in `outputs/day-11/traces-v3.jsonl`, and why does nothing measured change? In a product, what would you log next to every answer so that you could tell later which setting produced it?
2. **Next only, or the whole parent.** Rerun section 5 with `SETTING = "next"` and with `SETTING = "parent"` (put the line right after the rule in section 3.1). Which of the three held-out misses does each reach, and how many words does each send? Why does `parent` find fewer on dev at the same budget?
3. **Expand only when the hits agree** (auto-merging). Add a document's other sections only when at least two of the three hits come from it. How many words does that save on dev, and which answers does it keep? Does it reach the taxi question?
4. **Follow the reference.** The travel policy's *Safety while travelling* says "as described in the Security Incident Reporting policy". Write `linked(chunk_id)`, which returns the documents a section names by title, and add the best-ranked readable section of each linked document to the context. Does it reach *How to report* for the stolen phone? What would stop a document from linking to everything?
5. **A tighter budget.** Rerun sections 4 and 5 with `BUDGET = 150`. Which strategy loses answers first, and why?

LEARN.md's exercise asks you to find a corpus where the expansion *does* beat its control.